# SBOM analysis: extracting and querying package dependencies across image layers

This notebook walks through generating an SBOM with Syft for a multi-layer container image, then
parsing the Syft JSON output to explore which packages live in which layer and how dependencies
are distributed across the image. The goal is to be able to answer questions like:

- What Python packages are in this image?
- Which layer added `openssl`?
- How many OS-level packages vs language-level packages are there?

This is one way to approach SBOM analysis; Syft's JSON output is the primary surface, so the
analysis here focuses on wrangling that data.

In [ ]:
import json
import subprocess
import sys
from collections import Counter, defaultdict

In [ ]:
def generate_sbom(image: str, output_path: str = "/tmp/sbom.json") -> dict:
    """Run syft on an image and return the parsed JSON SBOM."""
    cmd = ["syft", image, "-o", "json", "--file", output_path]
    print(f"Generating SBOM for {image}...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        # syft --file writes output to the file instead of stdout
        # but we try fallback to stdout if --file wasn't supported
        if result.stdout:
            return json.loads(result.stdout)
        print(f"syft stderr: {result.stderr}", file=sys.stderr)
        raise RuntimeError(f"syft failed with exit code {result.returncode}")
    with open(output_path) as f:
        return json.load(f)

In [ ]:
# Generate SBOM for a multi-layer image — node:18-alpine has a good mix
# of OS (apk) and language (npm) packages across layers
image = "node:18-alpine"
sbom = generate_sbom(image)
print(f"SBOM artifact count: {len(sbom.get('artifacts', []))}")
print(f"Source: {sbom.get('source', {}).get('target', {}).get('userInput', 'unknown')}")

In [ ]:
def packages_per_layer(artifacts: list) -> dict:
    """Group packages by layer index."""
    layers: dict[int, list] = defaultdict(list)
    for pkg in artifacts:
        locations = pkg.get("locations", [])
        if locations:
            # Syft reports the layer index for each file location
            layer_idx = locations[0].get("layerIndex", -1)
        else:
            layer_idx = -1
        layers[layer_idx].append(pkg)
    return dict(sorted(layers.items()))

artifacts = sbom.get("artifacts", [])
by_layer = packages_per_layer(artifacts)

for layer_idx, pkgs in by_layer.items():
    print(f"Layer {layer_idx}: {len(pkgs)} packages")

total = sum(len(v) for v in by_layer.values())
print(f"\nTotal packages accounted for: {total}")

In [ ]:
def query_by_type(artifacts: list, pkg_type: str) -> list:
    """Filter artifacts by package type (e.g. 'apk', 'npm', 'python')."""
    return [p for p in artifacts if p.get("type") == pkg_type]

def query_by_name(artifacts: list, name_substring: str) -> list:
    """Find packages whose name contains the given substring."""
    return [p for p in artifacts if name_substring.lower() in p.get("name", "").lower()]

def query_by_layer(artifacts: list, layer_index: int) -> list:
    """Find all packages whose first location is in a specific layer."""
    result = []
    for p in artifacts:
        locations = p.get("locations", [])
        if locations and locations[0].get("layerIndex") == layer_index:
            result.append(p)
    return result

def package_type_counts(artifacts: list) -> dict:
    """Return a count of packages grouped by type."""
    return dict(Counter(p.get("type", "unknown") for p in artifacts))

In [ ]:
# --- Queries ---

print("=== Package type distribution ===")
for pkg_type, count in sorted(package_type_counts(artifacts).items()):
    print(f"  {pkg_type}: {count}")

print("\n=== npm packages (top 10) ===")
npm_pkgs = query_by_type(artifacts, "npm")
for p in npm_pkgs[:10]:
    print(f"  {p['name']}@{p.get('version', '?')}")
if len(npm_pkgs) > 10:
    print(f"  ... and {len(npm_pkgs) - 10} more")

print("\n=== Packages containing 'openssl' ===")
for p in query_by_name(artifacts, "openssl"):
    print(f"  {p['name']}@{p.get('version', '?')} ({p.get('type', '?')})")

print("\n=== Packages in first layer (layer 0) ===")
for p in query_by_layer(artifacts, 0)[:5]:
    print(f"  {p['name']}@{p.get('version', '?')} ({p.get('type', '?')})")

In [ ]:
# --- Cross-layer dependency analysis ---
# Some packages (like libcrypto) appear in multiple layers.
# Syft deduplicates by package ID — so each unique package appears once
# even if its files span layers. The first location determines the layer.

name_layer_map = defaultdict(set)
for p in artifacts:
    locs = p.get("locations", [])
    for loc in locs:
        li = loc.get("layerIndex", -1)
        if li >= 0:
            name_layer_map[p["name"]].add(li)

multi_layer = {name: layers for name, layers in name_layer_map.items() if len(layers) > 1}
print(f"Packages whose files span multiple layers: {len(multi_layer)}")
for name, layers in sorted(multi_layer.items())[:10]:
    print(f"  {name}: layers {sorted(layers)}")

## Verification

To verify the analysis:

1. Compare the package count with `syft <image> -o json | jq '.artifacts | length'`
2. Cross-check a specific package version with `apk info -a <pkg>` (for Alpine-based images)
3. Run the same notebook against a known image (e.g. `alpine:latest`) to validate the layer
   parsing works — Alpine has only OS-level packages, so `package_type_counts` should show
   only `apk` entries

The queries above cover the most common patterns: filtering by type, searching by name, and
identifying which layer introduced a package. For more advanced use cases, Syft also exposes
CPEs, licenses, and metadata fields that aren't explored here.